In [ ]:
# Import library
import torch, torch.nn as nn
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
torch.manual_seed(123)

In [ ]:
# Initial dataset
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
Xtr, Xte, ytr, yte = train_test_split(X, y, random_state=42, test_size= 0.2)

Xtr = torch.tensor(Xtr, dtype=torch.float32)
ytr = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)
Xte = torch.tensor(Xte, dtype=torch.float32)
yte = torch.tensor(yte, dtype=torch.float32).unsqueeze(1)

In [ ]:
loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)

In [ ]:
#Define class model
class Model(nn.Module):
  def __init__(self, input_dim, hidden=16):
    super().__init__()
    self.fc1 = nn.Linear(input_dim, hidden)
    self.fc2 = nn.Linear(hidden, 1)
  def forward(self, x):
    x = torch.relu(self.fc1(x))
    return self.fc2(x)

In [ ]:
# Evaluation Function
def evaluation(model, X, y):
  model.eval()
  with torch.no_grad():
    y_pred = model(X)

In [ ]:
model = Model(2,16)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [ ]:
for epoch in range(300):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
model.eval()
with torch.no_grad():
    acc = ((torch.sigmoid(model(Xte)) >= 0.5).float() == yte).float().mean()
print(f"test accuracy: {acc.item():.3f}")

test accuracy: 0.985
